In [ ]:
# 导入必要的库
import os
import geopandas as gpd
import rasterio
from rasterio.features import rasterize
from rasterio.transform import from_origin
import numpy as np
import glob

# 设置输入文件夹路径
input_folder = r"E:\chuli\over\area"
output_folder = input_folder  # 输出到同一文件夹

# 检查文件夹是否存在
if not os.path.exists(input_folder):
    raise FileNotFoundError(f"文件夹不存在: {input_folder}")

# 获取文件夹中所有的shp文件
shp_files = glob.glob(os.path.join(input_folder, "*.shp"))
print(f"找到 {len(shp_files)} 个shp文件:")
for i, shp_file in enumerate(shp_files):
    print(f"{i+1}. {os.path.basename(shp_file)}")

# 检查是否至少有两个shp文件
if len(shp_files) < 2:
    raise ValueError(f"需要至少2个shp文件，但只找到 {len(shp_files)} 个")

# 限制处理前两个文件
shp_files = shp_files[:2]
print(f"\n将处理前2个shp文件")

def vector_to_raster_with_projection(vector_path, output_path, pixel_size=10):
    """
    将矢量数据转换为栅格数据（掩膜），自动处理坐标系转换
    
    Parameters:
    vector_path: 矢量文件路径
    output_path: 输出栅格文件路径
    pixel_size: 像素大小（米）
    """
    # 读取shapefile
    print(f"正在读取: {os.path.basename(vector_path)}")
    gdf = gpd.read_file(vector_path)
    
    # 检查原始坐标系
    original_crs = gdf.crs
    print(f"原始坐标系: {original_crs}")
    
    # 如果是地理坐标系（单位是度），需要转换为投影坐标系
    if gdf.crs.is_geographic:
        print("检测到地理坐标系，需要转换为投影坐标系...")
        
        # 根据数据的中心点选择合适的UTM区域
        bounds = gdf.total_bounds
        center_lon = (bounds[0] + bounds[2]) / 2  # minx + maxx / 2
        center_lat = (bounds[1] + bounds[3]) / 2  # miny + maxy / 2
        
        # 计算UTM区域
        utm_zone = int((center_lon + 180) / 6) + 1
        if center_lat >= 0:
            # 北半球
            target_crs = f"EPSG:326{utm_zone:02d}"
        else:
            # 南半球
            target_crs = f"EPSG:327{utm_zone:02d}"
        
        print(f"中心点: ({center_lon:.4f}, {center_lat:.4f})")
        print(f"选择UTM区域: {target_crs}")
        
        # 转换坐标系
        gdf = gdf.to_crs(target_crs)
        print(f"已转换到坐标系: {gdf.crs}")
    else:
        print("数据已经是投影坐标系")
    
    # 获取转换后的边界框
    bounds = gdf.total_bounds
    minx, miny, maxx, maxy = bounds
    
    print(f"边界框: minx={minx:.2f}, miny={miny:.2f}, maxx={maxx:.2f}, maxy={maxy:.2f}")
    
    # 计算栅格尺寸（确保至少为1x1）
    width = max(1, int((maxx - minx) / pixel_size))
    height = max(1, int((maxy - miny) / pixel_size))
    
    print(f"计算的栅格尺寸: {width} x {height}")
    print(f"实际地理范围: {(maxx - minx):.2f}m x {(maxy - miny):.2f}m")
    
    # 创建仿射变换矩阵
    transform = from_origin(minx, maxy, pixel_size, pixel_size)
    
    # 将几何体转换为栅格
    shapes = [(geom, 1) for geom in gdf.geometry if geom is not None and not geom.is_empty]
    
    if not shapes:
        print(f"警告: {os.path.basename(vector_path)} 中没有有效的几何体")
        # 创建空栅格
        mask_raster = np.zeros((height, width), dtype=np.uint8)
    else:
        # 使用rasterize函数将矢量数据栅格化
        mask_raster = rasterize(
            shapes=shapes,
            out_shape=(height, width),
            transform=transform,
            fill=0,
            default_value=1,
            dtype=np.uint8
        )
    
    # 保存栅格文件
    with rasterio.open(
        output_path,
        'w',
        driver='GTiff',
        height=height,
        width=width,
        count=1,
        dtype=rasterio.uint8,
        crs=gdf.crs,
        transform=transform,
        nodata=0,
        compress='lzw',
        tiled=True
    ) as dst:
        dst.write(mask_raster, 1)
    
    print(f"✓ 已保存栅格文件: {os.path.basename(output_path)}")
    print(f"  栅格尺寸: {width} x {height}")
    print(f"  像素大小: {pixel_size}m x {pixel_size}m")
    print(f"  坐标系: {gdf.crs}")
    
    return output_path

def process_two_shapefiles_with_projection(shp_files, output_folder, pixel_size=10):
    """
    处理两个shapefile，分别转换为单独的栅格文件（带坐标系转换）
    """
    results = []
    
    for i, shp_file in enumerate(shp_files):
        # 生成输出文件名
        base_name = os.path.splitext(os.path.basename(shp_file))[0]
        output_raster = os.path.join(output_folder, f"{base_name}_mask.tif")
        
        print(f"\n{'='*50}")
        print(f"处理第 {i+1} 个文件 (共2个):")
        print(f"输入: {os.path.basename(shp_file)}")
        print(f"输出: {os.path.basename(output_raster)}")
        print(f"{'='*50}")
        
        try:
            # 转换为栅格（带投影转换）
            result = vector_to_raster_with_projection(shp_file, output_raster, pixel_size)
            results.append(result)
            print(f"✓ 第 {i+1} 个文件处理完成")
        except Exception as e:
            print(f"✗ 第 {i+1} 个文件处理失败: {str(e)}")
            import traceback
            traceback.print_exc()  # 打印详细错误信息
            continue
    
    return results

# 设置像素大小
pixel_size = 10  # 10m x 10m

# 处理两个shp文件
print(f"开始处理 {len(shp_files)} 个shapefile文件...")
print(f"目标像素大小: {pixel_size}m x {pixel_size}m")
print(f"输出文件夹: {output_folder}")

results = process_two_shapefiles_with_projection(shp_files, output_folder, pixel_size)

print(f"\n{'='*60}")
print("处理完成总结:")
print(f"{'='*60}")
print(f"成功处理了 {len(results)} 个文件")
for i, result in enumerate(results, 1):
    print(f"{i}. {os.path.basename(result)}")

# 验证生成的栅格文件
print(f"\n{'='*60}")
print("生成的栅格文件信息:")
print(f"{'='*60}")

for result in results:
    if os.path.exists(result):
        with rasterio.open(result) as src:
            print(f"文件: {os.path.basename(result)}")
            print(f"  尺寸: {src.width} x {src.height}")
            print(f"  像素大小: {src.res[0]:.2f}m x {src.res[1]:.2f}m")
            print(f"  坐标系: {src.crs}")
            print(f"  数据类型: {src.dtypes[0]}")
            
            # 读取数据统计信息
            data = src.read(1)
            unique_values = np.unique(data)
            print(f"  唯一值: {unique_values}")
            print(f"  非零像素数: {np.count_nonzero(data)}")
            print(f"  总像素数: {data.size}")
            print("-" * 40)
    else:
        print(f"文件不存在: {result}")

# 额外信息：显示原始shp文件信息
print(f"\n{'='*60}")
print("原始shp文件信息:")
print(f"{'='*60}")

for i, shp_file in enumerate(shp_files, 1):
    print(f"文件 {i}: {os.path.basename(shp_file)}")
    try:
        gdf = gpd.read_file(shp_file)
        print(f"  几何类型: {gdf.geometry.type.unique()}")
        print(f"  记录数: {len(gdf)}")
        print(f"  原始坐标系: {gdf.crs}")
        print(f"  边界: {gdf.total_bounds}")
        print(f"  是否为地理坐标系: {gdf.crs.is_geographic}")
        print("-" * 40)
    except Exception as e:
        print(f"  读取失败: {str(e)}")
        import traceback
        traceback.print_exc()


In [ ]:
import os
import rasterio
from rasterio.warp import calculate_default_transform, reproject, Resampling
import numpy as np

def clip_raster_with_mask(input_raster_path, mask_raster_path, output_path):
    """
    使用掩膜裁剪栅格数据
    """
    print(f"开始处理: {input_raster_path}")
    print(f"使用掩膜: {mask_raster_path}")
    
    # 读取掩膜文件获取目标信息
    with rasterio.open(mask_raster_path) as mask_src:
        mask_data = mask_src.read(1)  # 读取第一个波段
        mask_transform = mask_src.transform
        mask_crs = mask_src.crs
        mask_shape = (mask_src.height, mask_src.width)
        mask_bounds = mask_src.bounds
        mask_dtype = mask_src.dtypes[0]
        
        print(f"掩膜信息:")
        print(f"  坐标系统: {mask_crs}")
        print(f"  边界: {mask_bounds}")
        print(f"  分辨率: {mask_transform[0]} x {abs(mask_transform[4])} m")
        print(f"  尺寸: {mask_shape[1]} x {mask_shape[0]} 像素")
    
    # 读取输入栅格
    with rasterio.open(input_raster_path) as src:
        src_crs = src.crs
        print(f"输入栅格信息:")
        print(f"  坐标系统: {src_crs}")
        print(f"  边界: {src.bounds}")
        print(f"  分辨率: {src.transform[0]} x {abs(src.transform[4])} m")
        print(f"  尺寸: {src.width} x {src.height} 像素")
        
        # 检查坐标系统是否相同
        if src_crs != mask_crs:
            print("坐标系统不同，需要进行重投影...")
        
        # 创建输出配置
        kwargs = src.meta.copy()
        kwargs.update({
            'crs': mask_crs,
            'transform': mask_transform,
            'width': mask_shape[1],  # 使用掩膜的宽度
            'height': mask_shape[0], # 使用掩膜的高度
            'driver': 'GTiff',
            'compress': 'lzw',
            'tiled': True,
            'bigtiff': 'if_safer'
        })
        
        # 读取并重采样数据
        print("正在重采样数据...")
        reprojected_data = np.empty((src.count, mask_shape[0], mask_shape[1]), dtype=src.dtypes[0])
        
        for i in range(1, src.count + 1):
            reproject(
                source=rasterio.band(src, i),
                destination=reprojected_data[i-1],
                src_transform=src.transform,
                src_crs=src.crs,
                dst_transform=mask_transform,
                dst_crs=mask_crs,
                resampling=Resampling.nearest
            )
    
    # 应用掩膜
    print("应用掩膜...")
    # 扩展掩膜到所有波段
    mask_3d = np.broadcast_to(mask_data[np.newaxis, :, :], 
                             (reprojected_data.shape[0], mask_shape[0], mask_shape[1]))
    
    # 将掩膜外的区域设为NoData值
    no_data_value = kwargs.get('nodata', 0)
    if no_data_value is None:
        no_data_value = 0
    
    # 创建掩膜：掩膜为0的地方设为NoData
    mask_condition = mask_3d == 0
    reprojected_data = np.where(mask_condition, no_data_value, reprojected_data)
    
    # 创建输出文件
    print(f"写入输出文件: {output_path}")
    with rasterio.open(output_path, 'w', **kwargs) as dst:
        for i in range(reprojected_data.shape[0]):
            dst.write(reprojected_data[i], i + 1)
    
    print(f"裁剪完成，结果保存到: {output_path}")

def main():
    # 定义文件路径
    base_folder = r"E:\chuli\over\area\cut_result"
    mask_file = os.path.join(base_folder, "HK_area_mask.tif")
    input_file = os.path.join(base_folder, "landscan-hd-china-v1.tif")
    output_file = os.path.join(base_folder, "landscan-hd-china-v1_hk_clipped.tif")
    
    # 检查输入文件是否存在
    if not os.path.exists(mask_file):
        print(f"错误: 掩膜文件不存在 - {mask_file}")
        return
    
    if not os.path.exists(input_file):
        print(f"错误: 输入文件不存在 - {input_file}")
        return
    
    print("开始裁剪处理...")
    print(f"输入栅格: {input_file}")
    print(f"掩膜文件: {mask_file}")
    print(f"输出文件: {output_file}")
    
    try:
        # 执行裁剪操作
        clip_raster_with_mask(input_file, mask_file, output_file)
        print("处理完成!")
        
    except Exception as e:
        print(f"处理过程中出现错误: {str(e)}")
        import traceback
        traceback.print_exc()

if __name__ == "__main__":
    main()


In [ ]:
import os
import rasterio
from rasterio.warp import reproject, Resampling
import numpy as np

def clip_raster_with_mask_simple(input_raster_path, mask_raster_path, output_path):
    """
    使用掩膜裁剪栅格数据 - 简化版本
    """
    print(f"开始处理: {input_raster_path}")
    print(f"使用掩膜: {mask_raster_path}")
    
    # 读取掩膜文件获取目标信息
    with rasterio.open(mask_raster_path) as mask_src:
        mask_data = mask_src.read(1)  # 读取第一个波段
        mask_transform = mask_src.transform
        mask_crs = mask_src.crs
        mask_shape = (mask_src.height, mask_src.width)
        
        print(f"掩膜信息:")
        print(f"  坐标系统: {mask_crs}")
        print(f"  边界: {mask_src.bounds}")
        print(f"  分辨率: {mask_transform[0]} x {abs(mask_transform[4])} m")
        print(f"  尺寸: {mask_shape[1]} x {mask_shape[0]} 像素")
    
    # 读取输入栅格
    with rasterio.open(input_raster_path) as src:
        print(f"输入栅格信息:")
        print(f"  坐标系统: {src.crs}")
        print(f"  边界: {src.bounds}")
        
        # 创建输出配置
        kwargs = {
            'driver': 'GTiff',
            'height': mask_shape[0],
            'width': mask_shape[1],
            'count': src.count,
            'dtype': src.dtypes[0],
            'crs': mask_crs,
            'transform': mask_transform,
            'compress': 'lzw',
            'tiled': True,
            'bigtiff': 'if_safer',
            'nodata': 0  # 设置NoData值
        }
        
        # 读取并重采样数据
        print("正在重采样数据...")
        reprojected_data = np.empty((src.count, mask_shape[0], mask_shape[1]), dtype=src.dtypes[0])
        
        for i in range(1, src.count + 1):
            reproject(
                source=rasterio.band(src, i),
                destination=reprojected_data[i-1],
                src_transform=src.transform,
                src_crs=src.crs,
                dst_transform=mask_transform,
                dst_crs=mask_crs,
                resampling=Resampling.nearest,
                num_threads=4
            )
    
    # 应用掩膜
    print("应用掩膜...")
    # 将掩膜外的区域设为NoData值
    no_data_value = 0
    for i in range(reprojected_data.shape[0]):
        reprojected_data[i][mask_data == 0] = no_data_value
    
    # 创建输出文件
    print(f"写入输出文件: {output_path}")
    with rasterio.open(output_path, 'w', **kwargs) as dst:
        for i in range(reprojected_data.shape[0]):
            dst.write(reprojected_data[i], i + 1)
    
    print(f"裁剪完成，结果保存到: {output_path}")

def main():
    # 定义文件路径
    base_folder = r"E:\chuli\over\area\cut_result"
    mask_file = os.path.join(base_folder, "NY_area_mask.tif")
    
    # 定义要处理的输入文件列表
    input_files = [
        "landscan-usa-2021-conus-day.tif",
        "landscan-usa-2021-conus-night.tif"
    ]
    
    # 检查掩膜文件是否存在
    if not os.path.exists(mask_file):
        print(f"错误: 掩膜文件不存在 - {mask_file}")
        return
    
    # 处理每个输入文件
    for input_filename in input_files:
        input_file = os.path.join(base_folder, input_filename)
        
        # 检查输入文件是否存在
        if not os.path.exists(input_file):
            print(f"警告: 输入文件不存在 - {input_file}，跳过...")
            continue
        
        # 生成输出文件名
        name_part = input_filename.replace('.tif', '')
        output_file = os.path.join(base_folder, f"{name_part}_ny_clipped.tif")
        
        print("="*60)
        print(f"处理文件: {input_filename}")
        print(f"输入栅格: {input_file}")
        print(f"掩膜文件: {mask_file}")
        print(f"输出文件: {output_file}")
        
        try:
            # 执行裁剪操作
            clip_raster_with_mask_simple(input_file, mask_file, output_file)
            print(f"处理 {input_filename} 完成!")
            
        except Exception as e:
            print(f"处理 {input_filename} 时出现错误: {str(e)}")
            import traceback
            traceback.print_exc()
    
    print("="*60)
    print("所有处理完成!")

if __name__ == "__main__":
    main()


In [ ]:
import os
import rasterio
import numpy as np

def calculate_average_rasters(raster1_path, raster2_path, output_path):
    """
    计算两个栅格文件的平均值
    """
    print(f"开始计算平均值...")
    print(f"栅格1: {raster1_path}")
    print(f"栅格2: {raster2_path}")
    print(f"输出文件: {output_path}")
    
    # 读取第一个栅格
    with rasterio.open(raster1_path) as src1:
        # 读取数据
        data1 = src1.read()  # 读取所有波段
        profile1 = src1.profile.copy()
        
        print(f"栅格1信息:")
        print(f"  尺寸: {src1.width} x {src1.height}")
        print(f"  波段数: {src1.count}")
        print(f"  数据类型: {src1.dtypes[0]}")
        print(f"  NoData值: {src1.nodata}")
        print(f"  坐标系统: {src1.crs}")
    
    # 读取第二个栅格
    with rasterio.open(raster2_path) as src2:
        # 读取数据
        data2 = src2.read()  # 读取所有波段
        profile2 = src2.profile.copy()
        
        print(f"栅格2信息:")
        print(f"  尺寸: {src2.width} x {src2.height}")
        print(f"  波段数: {src2.count}")
        print(f"  数据类型: {src2.dtypes[0]}")
        print(f"  NoData值: {src2.nodata}")
        print(f"  坐标系统: {src2.crs}")
    
    # 检查两个栅格的尺寸是否相同
    if data1.shape != data2.shape:
        raise ValueError(f"两个栅格的尺寸不匹配: {data1.shape} vs {data2.shape}")
    
    print(f"数据形状: {data1.shape}")
    
    # 设置NoData值
    nodata1 = profile1.get('nodata', 0)
    nodata2 = profile2.get('nodata', 0)
    
    # 创建掩膜来标识有效数据
    if nodata1 is not None:
        mask1 = data1 != nodata1
    else:
        mask1 = ~np.isnan(data1)
    
    if nodata2 is not None:
        mask2 = data2 != nodata2
    else:
        mask2 = ~np.isnan(data2)
    
    # 计算平均值
    print("计算平均值...")
    
    # 创建副本以避免修改原始数据
    data1_copy = data1.astype(np.float64)
    data2_copy = data2.astype(np.float64)
    
    # 将NoData值设为NaN，便于计算
    if nodata1 is not None:
        data1_copy[data1 == nodata1] = np.nan
    else:
        mask1 = ~np.isnan(data1_copy)
    
    if nodata2 is not None:
        data2_copy[data2 == nodata2] = np.nan
    else:
        mask2 = ~np.isnan(data2_copy)
    
    # 计算平均值
    # 创建一个数组来存储有效值的数量
    valid_count = np.zeros_like(data1_copy, dtype=np.int8)
    valid_count[mask1] = 1
    valid_count[mask2] = valid_count[mask2] + 1
    
    # 计算总和
    sum_data = np.add(data1_copy, data2_copy)
    
    # 计算平均值
    average_data = np.divide(sum_data, valid_count, 
                           out=np.full_like(sum_data, np.nan), 
                           where=valid_count > 0)
    
    # 将NaN值转换为合适的NoData值
    # 找到一个合适的NoData值（不在数据范围内）
    if profile1['dtype'] in [rasterio.uint8, rasterio.uint16, rasterio.uint32, 
                             rasterio.int8, rasterio.int16, rasterio.int32]:
        # 整数类型
        if profile1['dtype'] == rasterio.uint8:
            nodata_output = 255
        elif profile1['dtype'] == rasterio.uint16:
            nodata_output = 65535
        elif profile1['dtype'] == rasterio.uint32:
            nodata_output = 4294967295
        elif profile1['dtype'] == rasterio.int16:
            nodata_output = -32768
        elif profile1['dtype'] == rasterio.int32:
            nodata_output = -2147483648
        else:
            nodata_output = -9999
    else:
        # 浮点类型
        nodata_output = -9999.0
    
    # 将NaN值替换为NoData值
    average_data = np.nan_to_num(average_data, nan=nodata_output)
    
    # 转换回原始数据类型
    if profile1['dtype'] in [rasterio.uint8, rasterio.uint16, rasterio.uint32, 
                             rasterio.int8, rasterio.int16, rasterio.int32]:
        average_data = np.clip(average_data, np.iinfo(profile1['dtype']).min, 
                              np.iinfo(profile1['dtype']).max)
    
    average_data = average_data.astype(profile1['dtype'])
    
    # 更新profile用于输出
    profile1['nodata'] = nodata_output
    profile1['compress'] = 'lzw'
    profile1['tiled'] = True
    profile1['bigtiff'] = 'if_safer'
    
    # 写入输出文件
    print(f"写入输出文件: {output_path}")
    with rasterio.open(output_path, 'w', **profile1) as dst:
        dst.write(average_data)
    
    print(f"平均值计算完成，结果保存到: {output_path}")

def main():
    # 定义文件路径
    base_folder = r"E:\chuli\over\area\cut_result"
    
    # 输入文件路径
    raster1_file = os.path.join(base_folder, "landscan-usa-2021-conus-day_ny_clipped.tif")
    raster2_file = os.path.join(base_folder, "landscan-usa-2021-conus-night_ny_clipped.tif")
    
    # 输出文件路径
    output_file = os.path.join(base_folder, "NY_person_clipped.tif")
    
    # 检查输入文件是否存在
    if not os.path.exists(raster1_file):
        print(f"错误: 文件不存在 - {raster1_file}")
        return
    
    if not os.path.exists(raster2_file):
        print(f"错误: 文件不存在 - {raster2_file}")
        return
    
    print("开始计算两个栅格的平均值...")
    
    try:
        # 执行平均值计算
        calculate_average_rasters(raster1_file, raster2_file, output_file)
        print("平均值计算完成!")
        
    except Exception as e:
        print(f"计算过程中出现错误: {str(e)}")
        import traceback
        traceback.print_exc()

if __name__ == "__main__":
    main()
